## Recommender System Types

- Popularity based recommender system
  - Ex: Youtube trending, IMDB top 50
  - Give preference to popularity rather than user preference
  - Based on formula calculates popularity
- Content based recommender system
  - Content similarity based recommender system
- Collaborative filtering based recommender system
  - Based on user preference
- Hybrid based recommender system
  - Approach based on popularity , content and collaborative
  - Providing weightage on all other systems and generate new output based on the combined results

In [200]:
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity

In [11]:
books = pd.read_csv('../data/books.csv',on_bad_lines='skip',encoding='latin-1')
users = pd.read_csv('../data/users.csv',on_bad_lines='skip',encoding='latin-1')
ratings = pd.read_csv('../data/ratings.csv',on_bad_lines='skip',encoding='latin-1')

/var/folders/zc/fcrq35tj1y11zktl3108b93r0000gn/T/ipykernel_52321/3144973985.py:1: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv('../data/books.csv',on_bad_lines='skip',encoding='latin-1')


In [12]:
books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [13]:
users.head()

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [15]:
ratings.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [16]:
books.info()

<class 'pandas.DataFrame'>
RangeIndex: 271360 entries, 0 to 271359
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   ISBN                 271360 non-null  str   
 1   Book-Title           271360 non-null  str   
 2   Book-Author          271358 non-null  str   
 3   Year-Of-Publication  271360 non-null  object
 4   Publisher            271358 non-null  str   
 5   Image-URL-S          271360 non-null  str   
 6   Image-URL-M          271360 non-null  str   
 7   Image-URL-L          271357 non-null  str   
dtypes: object(1), str(7)
memory usage: 83.4+ MB


In [18]:
books.isnull().sum()

ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64

In [19]:
users.isnull().sum()

User-ID          0
Location         0
Age         110762
dtype: int64

In [22]:
users.isna().sum()

User-ID          0
Location         0
Age         110762
dtype: int64

In [23]:
ratings.isna().sum()

User-ID        0
ISBN           0
Book-Rating    0
dtype: int64

In [30]:
users.duplicated().sum()

np.int64(0)

In [31]:
ratings.duplicated().sum()

np.int64(0)

In [32]:
books.duplicated().sum()

np.int64(0)

In [33]:
books.shape

(271360, 8)

In [34]:
ratings.shape

(1149780, 3)

In [35]:
users.shape

(278858, 3)

## Popular based recommendation system

RULE: Top 50 books with highest average rating based on minimum 250 voting by users

In [36]:
ratings_with_name =  ratings.merge(books,on='ISBN') 

In [37]:
ratings_with_name.head()

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...
1,276726,0155061224,5,Rites of Passage,Judith Rae,2001,Heinle,http://images.amazon.com/images/P/0155061224.0...,http://images.amazon.com/images/P/0155061224.0...,http://images.amazon.com/images/P/0155061224.0...
2,276727,0446520802,0,The Notebook,Nicholas Sparks,1996,Warner Books,http://images.amazon.com/images/P/0446520802.0...,http://images.amazon.com/images/P/0446520802.0...,http://images.amazon.com/images/P/0446520802.0...
3,276729,052165615X,3,Help!: Level 1,Philip Prowse,1999,Cambridge University Press,http://images.amazon.com/images/P/052165615X.0...,http://images.amazon.com/images/P/052165615X.0...,http://images.amazon.com/images/P/052165615X.0...
4,276729,0521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,2001,Cambridge University Press,http://images.amazon.com/images/P/0521795028.0...,http://images.amazon.com/images/P/0521795028.0...,http://images.amazon.com/images/P/0521795028.0...


In [99]:
ratings_with_name.shape

(1031136, 10)

In [110]:
ratings_with_name.loc[ratings_with_name['Book-Title'] == 'A Light in the Storm: The Civil War Diary of Amelia Martin, Fenwick Island, Delaware, 1861 (Dear America)']

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L


In [111]:
ratings_with_name.groupby('Book-Title')['Book-Rating'].count()

Book-Title
 A Light in the Storm: The Civil War Diary of Amelia Martin, Fenwick Island, Delaware, 1861 (Dear America)    4
 Always Have Popsicles                                                                                        1
 Apple Magic (The Collector's series)                                                                         1
 Ask Lily (Young Women of Faith: Lily Series, Book 5)                                                         1
 Beyond IBM: Leadership Marketing and Finance for the 1990s                                                   1
                                                                                                             ..
Ã?Ã?lpiraten.                                                                                               2
Ã?Ã?rger mit Produkt X. Roman.                                                                              4
Ã?Ã?sterlich leben.                                                                        

In [127]:
num_rating_df = ratings_with_name.groupby('Book-Title')['Book-Rating'].count().reset_index()

In [128]:
num_rating_df.rename(columns={'Book-Rating':'total_num_of_rating'},inplace=True)

In [129]:
num_rating_df

,Book-Title,total_num_of_rating
0,A Light in the Storm: The Civil War Diary of ...,4
1,Always Have Popsicles,1
2,Apple Magic (The Collector's series),1
3,"Ask Lily (Young Women of Faith: Lily Series, ...",1
4,Beyond IBM: Leadership Marketing and Finance ...,1
...,...,...
241066,Ã?Ã?lpiraten.,2
241067,Ã?Ã?rger mit Produkt X. Roman.,4
241068,Ã?Ã?sterlich leben.,1
241069,Ã?Ã?stlich der Berge.,3


In [130]:
num_rating_df.iloc[0]['Book-Title']

' A Light in the Storm: The Civil War Diary of Amelia Martin, Fenwick Island, Delaware, 1861 (Dear America)'

In [131]:
average_rating_df = round(ratings_with_name.groupby('Book-Title')['Book-Rating'].mean().reset_index(),2)
average_rating_df.rename(columns={'Book-Rating':'average_num_of_rating'},inplace=True)
average_rating_df

,Book-Title,average_num_of_rating
0,A Light in the Storm: The Civil War Diary of ...,2.25
1,Always Have Popsicles,0.00
2,Apple Magic (The Collector's series),0.00
3,"Ask Lily (Young Women of Faith: Lily Series, ...",8.00
4,Beyond IBM: Leadership Marketing and Finance ...,0.00
...,...,...
241066,Ã?Ã?lpiraten.,0.00
241067,Ã?Ã?rger mit Produkt X. Roman.,5.25
241068,Ã?Ã?sterlich leben.,7.00
241069,Ã?Ã?stlich der Berge.,2.67


In [132]:
popularity_df = num_rating_df.merge(average_rating_df,on='Book-Title')
popularity_df

,Book-Title,total_num_of_rating,average_num_of_rating
0,A Light in the Storm: The Civil War Diary of ...,4,2.25
1,Always Have Popsicles,1,0.00
2,Apple Magic (The Collector's series),1,0.00
3,"Ask Lily (Young Women of Faith: Lily Series, ...",1,8.00
4,Beyond IBM: Leadership Marketing and Finance ...,1,0.00
...,...,...,...
241066,Ã?Ã?lpiraten.,2,0.00
241067,Ã?Ã?rger mit Produkt X. Roman.,4,5.25
241068,Ã?Ã?sterlich leben.,1,7.00
241069,Ã?Ã?stlich der Berge.,3,2.67


In [133]:
popularity_df = popularity_df[popularity_df['total_num_of_rating']>250]

In [134]:
popularity_df 

,Book-Title,total_num_of_rating,average_num_of_rating
764,1984,284,4.45
818,1st to Die: A Novel,509,3.58
1048,2nd Chance,356,3.27
1760,A Bend in the Road,346,3.36
2281,"A Child Called \It\"": One Child's Courage to S...",265,4.09
...,...,...,...
233850,White Oleander : A Novel,387,3.51
233851,White Oleander : A Novel (Oprah's Book Club),356,3.77
234740,Wicked: The Life and Times of the Wicked Witch...,326,3.77
234951,Wild Animus,2502,1.02


In [135]:
popularity_df.sort_values('average_num_of_rating',ascending=False,inplace=True)

In [137]:
popularity_df = popularity_df.head(50)
popularity_df 

,Book-Title,total_num_of_rating,average_num_of_rating
80434,Harry Potter and the Prisoner of Azkaban (Book 3),428,5.85
80422,Harry Potter and the Goblet of Fire (Book 4),387,5.82
80441,Harry Potter and the Sorcerer's Stone (Book 1),278,5.74
80426,Harry Potter and the Order of the Phoenix (Boo...,347,5.50
80414,Harry Potter and the Chamber of Secrets (Book 2),556,5.18
191612,The Hobbit : The Enchanting Prelude to The Lor...,281,5.01
187377,The Fellowship of the Ring (The Lord of the Ri...,368,4.95
80445,Harry Potter and the Sorcerer's Stone (Harry P...,575,4.90
211384,"The Two Towers (The Lord of the Rings, Part 2)",260,4.88
219741,To Kill a Mockingbird,510,4.70


In [138]:
popularity_df.shape

(50, 3)

In [140]:
popularity_df = popularity_df.merge(books,on='Book-Title')
popularity_df.sample(10)

,Book-Title,total_num_of_rating,average_num_of_rating,ISBN,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
50,The Da Vinci Code,898,4.64,0385504217,Dan Brown,2003,Doubleday Books,http://images.amazon.com/images/P/0385504217.0...,http://images.amazon.com/images/P/0385504217.0...,http://images.amazon.com/images/P/0385504217.0...
96,The Hitchhiker's Guide to the Galaxy,268,4.33,1590072561,Douglas Adams,2002,New Millennium Press,http://images.amazon.com/images/P/1590072561.0...,http://images.amazon.com/images/P/1590072561.0...,http://images.amazon.com/images/P/1590072561.0...
195,Wicked: The Life and Times of the Wicked Witch...,326,3.77,0060987103,Gregory Maguire,1996,Regan Books,http://images.amazon.com/images/P/0060987103.0...,http://images.amazon.com/images/P/0060987103.0...,http://images.amazon.com/images/P/0060987103.0...
184,Suzanne's Diary for Nicholas,457,3.82,0446611085,James Patterson,2003,Warner Books,http://images.amazon.com/images/P/0446611085.0...,http://images.amazon.com/images/P/0446611085.0...,http://images.amazon.com/images/P/0446611085.0...
46,To Kill a Mockingbird,510,4.70,0899668585,Harper Lee,1991,Buccaneer Books Inc,http://images.amazon.com/images/P/0899668585.0...,http://images.amazon.com/images/P/0899668585.0...,http://images.amazon.com/images/P/0899668585.0...
61,The Catcher in the Rye,449,4.55,0808514032,J.D. Salinger,1999,Sagebrush Bound,http://images.amazon.com/images/P/0808514032.0...,http://images.amazon.com/images/P/0808514032.0...,http://images.amazon.com/images/P/0808514032.0...
34,"The Two Towers (The Lord of the Rings, Part 2)",260,4.88,0618260277,J. R. R. Tolkien,2002,Houghton Mifflin Company,http://images.amazon.com/images/P/0618260277.0...,http://images.amazon.com/images/P/0618260277.0...,http://images.amazon.com/images/P/0618260277.0...
162,The Color Purple,314,3.96,0521403979,Alice Walker,1990,Cambridge University Press,http://images.amazon.com/images/P/0521403979.0...,http://images.amazon.com/images/P/0521403979.0...,http://images.amazon.com/images/P/0521403979.0...
74,Neverwhere,265,4.45,1565112318,Neil Gaiman,1997,Highbridge Audio,http://images.amazon.com/images/P/1565112318.0...,http://images.amazon.com/images/P/1565112318.0...,http://images.amazon.com/images/P/1565112318.0...
145,Skipping Christmas,322,4.01,0739301985,JOHN GRISHAM,2002,Random House Audio,http://images.amazon.com/images/P/0739301985.0...,http://images.amazon.com/images/P/0739301985.0...,http://images.amazon.com/images/P/0739301985.0...


In [141]:
popularity_df.shape

(196, 10)

In [142]:
popularity_df[popularity_df['total_num_of_rating']<250]

,Book-Title,total_num_of_rating,average_num_of_rating,ISBN,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L


In [143]:
popularity_df =  popularity_df.drop_duplicates('Book-Title')
popularity_df.head()

,Book-Title,total_num_of_rating,average_num_of_rating,ISBN,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,Harry Potter and the Prisoner of Azkaban (Book 3),428,5.85,0439136350,J. K. Rowling,1999,Scholastic,http://images.amazon.com/images/P/0439136350.0...,http://images.amazon.com/images/P/0439136350.0...,http://images.amazon.com/images/P/0439136350.0...
3,Harry Potter and the Goblet of Fire (Book 4),387,5.82,0439139597,J. K. Rowling,2000,Scholastic,http://images.amazon.com/images/P/0439139597.0...,http://images.amazon.com/images/P/0439139597.0...,http://images.amazon.com/images/P/0439139597.0...
5,Harry Potter and the Sorcerer's Stone (Book 1),278,5.74,0590353403,J. K. Rowling,1998,Scholastic,http://images.amazon.com/images/P/0590353403.0...,http://images.amazon.com/images/P/0590353403.0...,http://images.amazon.com/images/P/0590353403.0...
9,Harry Potter and the Order of the Phoenix (Boo...,347,5.50,043935806X,J. K. Rowling,2003,Scholastic,http://images.amazon.com/images/P/043935806X.0...,http://images.amazon.com/images/P/043935806X.0...,http://images.amazon.com/images/P/043935806X.0...
13,Harry Potter and the Chamber of Secrets (Book 2),556,5.18,0439064872,J. K. Rowling,2000,Scholastic,http://images.amazon.com/images/P/0439064872.0...,http://images.amazon.com/images/P/0439064872.0...,http://images.amazon.com/images/P/0439064872.0...


In [145]:
popularity_df.shape

(50, 10)

In [146]:
popularity_df.columns

Index(['Book-Title', 'total_num_of_rating', 'average_num_of_rating', 'ISBN',
       'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-S',
       'Image-URL-M', 'Image-URL-L'],
      dtype='str')

In [147]:
popularity_df = popularity_df[['Book-Title', 'total_num_of_rating', 'average_num_of_rating','Publisher','Image-URL-M','Book-Author']]
popularity_df 

,Book-Title,total_num_of_rating,average_num_of_rating,Publisher,Image-URL-M,Book-Author
0,Harry Potter and the Prisoner of Azkaban (Book 3),428,5.85,Scholastic,http://images.amazon.com/images/P/0439136350.0...,J. K. Rowling
3,Harry Potter and the Goblet of Fire (Book 4),387,5.82,Scholastic,http://images.amazon.com/images/P/0439139597.0...,J. K. Rowling
5,Harry Potter and the Sorcerer's Stone (Book 1),278,5.74,Scholastic,http://images.amazon.com/images/P/0590353403.0...,J. K. Rowling
9,Harry Potter and the Order of the Phoenix (Boo...,347,5.50,Scholastic,http://images.amazon.com/images/P/043935806X.0...,J. K. Rowling
13,Harry Potter and the Chamber of Secrets (Book 2),556,5.18,Scholastic,http://images.amazon.com/images/P/0439064872.0...,J. K. Rowling
16,The Hobbit : The Enchanting Prelude to The Lor...,281,5.01,Del Rey,http://images.amazon.com/images/P/0345339681.0...,J.R.R. TOLKIEN
17,The Fellowship of the Ring (The Lord of the Ri...,368,4.95,Del Rey,http://images.amazon.com/images/P/0345339703.0...,J.R.R. TOLKIEN
26,Harry Potter and the Sorcerer's Stone (Harry P...,575,4.90,Arthur A. Levine Books,http://images.amazon.com/images/P/059035342X.0...,J. K. Rowling
28,"The Two Towers (The Lord of the Rings, Part 2)",260,4.88,Del Rey,http://images.amazon.com/images/P/0345339711.0...,J.R.R. TOLKIEN
39,To Kill a Mockingbird,510,4.70,Little Brown &amp; Company,http://images.amazon.com/images/P/0446310786.0...,Harper Lee


In [148]:
popularity_df.shape

(50, 6)

In [149]:
popularity_df.isnull().sum()

Book-Title               0
total_num_of_rating      0
average_num_of_rating    0
Publisher                0
Image-URL-M              0
Book-Author              0
dtype: int64

In [150]:
popularity_df.duplicated().sum()

np.int64(0)

## Collaborative filtering based recommender system

In [171]:
ratings_with_name.head(5)

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...
1,276726,0155061224,5,Rites of Passage,Judith Rae,2001,Heinle,http://images.amazon.com/images/P/0155061224.0...,http://images.amazon.com/images/P/0155061224.0...,http://images.amazon.com/images/P/0155061224.0...
2,276727,0446520802,0,The Notebook,Nicholas Sparks,1996,Warner Books,http://images.amazon.com/images/P/0446520802.0...,http://images.amazon.com/images/P/0446520802.0...,http://images.amazon.com/images/P/0446520802.0...
3,276729,052165615X,3,Help!: Level 1,Philip Prowse,1999,Cambridge University Press,http://images.amazon.com/images/P/052165615X.0...,http://images.amazon.com/images/P/052165615X.0...,http://images.amazon.com/images/P/052165615X.0...
4,276729,0521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,2001,Cambridge University Press,http://images.amazon.com/images/P/0521795028.0...,http://images.amazon.com/images/P/0521795028.0...,http://images.amazon.com/images/P/0521795028.0...


In [172]:
ratings_with_name.shape

(1031136, 10)

In [173]:
ratings_with_name[ratings_with_name['User-ID']==2]

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
8429,2,0195153448,0,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...


In [174]:
x = ratings_with_name.groupby('User-ID')['Book-Rating'].count()>200

In [175]:
x[x]

User-ID
254       True
2276      True
2766      True
2977      True
3363      True
          ... 
274308    True
275970    True
277427    True
277639    True
278418    True
Name: Book-Rating, Length: 811, dtype: bool

In [176]:
user_ids = x[x].index

In [177]:
user_ids

Index([   254,   2276,   2766,   2977,   3363,   4017,   4385,   6251,   6323,
         6543,
       ...
       271705, 273979, 274004, 274061, 274301, 274308, 275970, 277427, 277639,
       278418],
      dtype='int64', name='User-ID', length=811)

In [178]:
ratings_with_name[ratings_with_name['User-ID'].isin(user_ids)]

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
1150,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...,http://images.amazon.com/images/P/002542730X.0...,http://images.amazon.com/images/P/002542730X.0...
1151,277427,0026217457,0,Vegetarian Times Complete Cookbook,Lucy Moll,1995,John Wiley &amp; Sons,http://images.amazon.com/images/P/0026217457.0...,http://images.amazon.com/images/P/0026217457.0...,http://images.amazon.com/images/P/0026217457.0...
1152,277427,003008685X,8,Pioneers,James Fenimore Cooper,1974,Thomson Learning,http://images.amazon.com/images/P/003008685X.0...,http://images.amazon.com/images/P/003008685X.0...,http://images.amazon.com/images/P/003008685X.0...
1153,277427,0030615321,0,"Ask for May, Settle for June (A Doonesbury book)",G. B. Trudeau,1982,Henry Holt &amp; Co,http://images.amazon.com/images/P/0030615321.0...,http://images.amazon.com/images/P/0030615321.0...,http://images.amazon.com/images/P/0030615321.0...
1154,277427,0060002050,0,On a Wicked Dawn (Cynster Novels),Stephanie Laurens,2002,Avon Books,http://images.amazon.com/images/P/0060002050.0...,http://images.amazon.com/images/P/0060002050.0...,http://images.amazon.com/images/P/0060002050.0...
...,...,...,...,...,...,...,...,...,...,...
1029357,275970,1931868123,0,There's a Porcupine in My Outhouse: Misadventu...,Mike Tougias,2002,Capital Books (VA),http://images.amazon.com/images/P/1931868123.0...,http://images.amazon.com/images/P/1931868123.0...,http://images.amazon.com/images/P/1931868123.0...
1029358,275970,3411086211,10,Die Biene.,Sybil GrÃ?ÃÂ¤fin SchÃ?ÃÂ¶nfeldt,1993,"Bibliographisches Institut, Mannheim",http://images.amazon.com/images/P/3411086211.0...,http://images.amazon.com/images/P/3411086211.0...,http://images.amazon.com/images/P/3411086211.0...
1029359,275970,3829021860,0,The Penis Book,Joseph Cohen,1999,Konemann,http://images.amazon.com/images/P/3829021860.0...,http://images.amazon.com/images/P/3829021860.0...,http://images.amazon.com/images/P/3829021860.0...
1029360,275970,4770019572,0,Musashi,Eiji Yoshikawa,1995,Kodansha International (JPN),http://images.amazon.com/images/P/4770019572.0...,http://images.amazon.com/images/P/4770019572.0...,http://images.amazon.com/images/P/4770019572.0...


In [179]:
ratings_with_name[ratings_with_name['User-ID'].isin(user_ids)].shape

(474007, 10)

In [180]:
filtered_ratings_df = ratings_with_name[ratings_with_name['User-ID'].isin(user_ids)]
filtered_ratings_df.shape

(474007, 10)

In [183]:
filtered_ratings_df.groupby('Book-Title').count()['Book-Rating'].reset_index()

,Book-Title,Book-Rating
0,A Light in the Storm: The Civil War Diary of ...,2
1,Always Have Popsicles,1
2,Apple Magic (The Collector's series),1
3,Beyond IBM: Leadership Marketing and Finance ...,1
4,Clifford Visita El Hospital (Clifford El Gran...,1
...,...,...
155840,Ã?Ã?ber das Fernsehen.,2
155841,Ã?Ã?ber die Pflicht zum Ungehorsam gegen den...,3
155842,Ã?Ã?lpiraten.,1
155843,Ã?Ã?stlich der Berge.,1


In [185]:
y = filtered_ratings_df.groupby('Book-Title').count()['Book-Rating']>=50

In [186]:
y[y]

Book-Title
1984                                                                 True
1st to Die: A Novel                                                  True
2nd Chance                                                           True
4 Blondes                                                            True
A Bend in the Road                                                   True
                                                                     ... 
Year of Wonders                                                      True
You Belong To Me                                                     True
Zen and the Art of Motorcycle Maintenance: An Inquiry into Values    True
Zoya                                                                 True
\O\" Is for Outlaw"                                                  True
Name: Book-Rating, Length: 706, dtype: bool

In [187]:
book_titles = y[y].index

In [188]:
book_titles 

Index(['1984', '1st to Die: A Novel', '2nd Chance', '4 Blondes',
       'A Bend in the Road', 'A Case of Need',
       'A Child Called \It\": One Child's Courage to Survive"',
       'A Civil Action', 'A Day Late and a Dollar Short', 'A Fine Balance',
       ...
       'Winter Solstice', 'Wish You Well', 'Without Remorse',
       'Wizard and Glass (The Dark Tower, Book 4)', 'Wuthering Heights',
       'Year of Wonders', 'You Belong To Me',
       'Zen and the Art of Motorcycle Maintenance: An Inquiry into Values',
       'Zoya', '\O\" Is for Outlaw"'],
      dtype='str', name='Book-Title', length=706)

In [190]:
final_book_ratings = filtered_ratings_df[filtered_ratings_df['Book-Title'].isin(book_titles)]

In [191]:
final_book_ratings 

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
1150,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...,http://images.amazon.com/images/P/002542730X.0...,http://images.amazon.com/images/P/002542730X.0...
1163,277427,0060930535,0,The Poisonwood Bible: A Novel,Barbara Kingsolver,1999,Perennial,http://images.amazon.com/images/P/0060930535.0...,http://images.amazon.com/images/P/0060930535.0...,http://images.amazon.com/images/P/0060930535.0...
1165,277427,0060934417,0,Bel Canto: A Novel,Ann Patchett,2002,Perennial,http://images.amazon.com/images/P/0060934417.0...,http://images.amazon.com/images/P/0060934417.0...,http://images.amazon.com/images/P/0060934417.0...
1168,277427,0061009059,9,One for the Money (Stephanie Plum Novels (Pape...,Janet Evanovich,1995,HarperTorch,http://images.amazon.com/images/P/0061009059.0...,http://images.amazon.com/images/P/0061009059.0...,http://images.amazon.com/images/P/0061009059.0...
1174,277427,006440188X,0,The Secret Garden,Frances Hodgson Burnett,1998,HarperTrophy,http://images.amazon.com/images/P/006440188X.0...,http://images.amazon.com/images/P/006440188X.0...,http://images.amazon.com/images/P/006440188X.0...
...,...,...,...,...,...,...,...,...,...,...
1029196,275970,1400031354,0,Tears of the Giraffe (No.1 Ladies Detective Ag...,Alexander McCall Smith,2002,Anchor,http://images.amazon.com/images/P/1400031354.0...,http://images.amazon.com/images/P/1400031354.0...,http://images.amazon.com/images/P/1400031354.0...
1029197,275970,1400031362,0,Morality for Beautiful Girls (No.1 Ladies Dete...,Alexander McCall Smith,2002,Anchor,http://images.amazon.com/images/P/1400031362.0...,http://images.amazon.com/images/P/1400031362.0...,http://images.amazon.com/images/P/1400031362.0...
1029270,275970,1573229725,0,Fingersmith,Sarah Waters,2002,Riverhead Books,http://images.amazon.com/images/P/1573229725.0...,http://images.amazon.com/images/P/1573229725.0...,http://images.amazon.com/images/P/1573229725.0...
1029309,275970,1586210661,9,Me Talk Pretty One Day,David Sedaris,2001,Time Warner Audio Major,http://images.amazon.com/images/P/1586210661.0...,http://images.amazon.com/images/P/1586210661.0...,http://images.amazon.com/images/P/1586210661.0...


In [193]:
final_book_ratings.duplicated().sum()

np.int64(0)

In [195]:
pi_table = final_book_ratings.pivot_table(index='Book-Title',columns=['User-ID'],values='Book-Rating')

In [196]:
pi_table.shape

(706, 810)

In [197]:
pi_table.fillna(0,inplace=True)

User-ID,254,2276,2766,2977,3363,4017,4385,6251,6323,6543,...,271705,273979,274004,274061,274301,274308,275970,277427,277639,278418
Book-Title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A Bend in the Road,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Year of Wonders,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
You Belong To Me,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zen and the Art of Motorcycle Maintenance: An Inquiry into Values,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [203]:
similarity_score = cosine_similarity(pi_table)

In [204]:
similarity_score

array([[1.        , 0.10255025, 0.01220856, ..., 0.12110367, 0.07347567,
        0.04316046],
       [0.10255025, 1.        , 0.2364573 , ..., 0.07446129, 0.16773875,
        0.14263397],
       [0.01220856, 0.2364573 , 1.        , ..., 0.04558758, 0.04938579,
        0.10796119],
       ...,
       [0.12110367, 0.07446129, 0.04558758, ..., 1.        , 0.07085128,
        0.0196177 ],
       [0.07347567, 0.16773875, 0.04938579, ..., 0.07085128, 1.        ,
        0.10602962],
       [0.04316046, 0.14263397, 0.10796119, ..., 0.0196177 , 0.10602962,
        1.        ]], shape=(706, 706))

In [205]:
cosine_similarity(pi_table).shape

(706, 706)

In [206]:
similarity_score.shape

(706, 706)

In [244]:
def recommend_book(book_title):
    suggested_books = []
    index_found =  np.where(pi_table.index == book_title)
    if len(index_found[0]) == 0:
        return "Books Not Found"
    book_index =   index_found[0][0]
    distances = sorted(list(enumerate(similarity_score[book_index])),key=lambda x:x[1],reverse=True)[1:6]
    for i in distances:
        suggested_books.append(pi_table.index[i[0]])
    return suggested_books
    

In [246]:
recommend_book('Message in a Bottle')

['Nights in Rodanthe',
 'The Mulberry Tree',
 'A Walk to Remember',
 "River's End",
 'Nightmares &amp; Dreamscapes']